In [7]:
import pandas as pd
import numpy as np
import torch
import pickle

from scipy.sparse import csr_matrix, hstack
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k
from sklearn.decomposition import TruncatedSVD

### Loading behaviours.tsv and converting to interactions matrix

In [6]:
beh_train = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'],
    dtype=str
)
beh_train['history'] = beh_train['history'].fillna('').str.split()


In [8]:
def expand_impressions(df):
    rows = []
    for uid, imp_list in zip(df['user_id'], df['impressions'].str.split()):
        for imp in imp_list:
            nid, lbl = imp.split('-')
            if lbl == '1':
                rows.append((uid, nid))
    return pd.DataFrame(rows, columns=['user_id','article_id'])


train_df = expand_impressions(beh_train)

In [10]:
beh_val = pd.read_csv(
    'MINDsmall_dev/behaviors.tsv', sep='\t', header=None,
    names=['impression_id','user_id','timestamp','history','impressions'], dtype=str
)
beh_val['history'] = beh_val['history'].fillna('').str.split()

In [12]:
val_df = expand_impressions(beh_val)


In [14]:
# 1.3) Remove overlaps between train and val
train_pairs = set(zip(train_df['user_id'], train_df['article_id']))
val_df = val_df[~val_df.apply(lambda r: (r['user_id'], r['article_id']) in train_pairs, axis=1)]
val_df = val_df.reset_index(drop=True)

In [16]:
print(f"Train clicks: {len(train_df):,}, Val clicks (filtered): {len(val_df):,}")


Train clicks: 236,344, Val clicks (filtered): 111,357


In [18]:
# 1.4) Fit Dataset on union of users/items
all_users = pd.Index(train_df['user_id']).union(val_df['user_id'])
all_items = pd.Index(train_df['article_id']).union(val_df['article_id'])

dataset = Dataset()
dataset.fit(users=all_users, items=all_items)


In [20]:
train_interactions, _ = dataset.build_interactions(
    train_df[['user_id','article_id']].itertuples(index=False, name=None)
)
val_interactions, _ = dataset.build_interactions(
    val_df[['user_id','article_id']].itertuples(index=False, name=None)
)

print(f"Interaction matrices: train {train_interactions.shape}, val {val_interactions.shape}")


Interaction matrices: train (94057, 9100), val (94057, 9100)


In [22]:
print(f"Interaction matrices built:")
print(f"  Train: {train_interactions.shape}")
print(f"  Val:   {val_interactions.shape}")

Interaction matrices built:
  Train: (94057, 9100)
  Val:   (94057, 9100)


### Building item features from BERT embeddings and category OHE

In [9]:
train_news = pd.read_csv(
    'MINDsmall_train/news.tsv',
    sep='\t', header=None,
    names=['newid','vertical','subvertical','title','abstract','url','ent_title','ent_abstract'],
    dtype=str
).set_index('newid')

# Load dev news metadata
dev_news = pd.read_csv(
    'MINDsmall_dev/news.tsv',
    sep='\t', header=None,
    names=['newid','vertical','subvertical','title','abstract','url','ent_title','ent_abstract'],
    dtype=str
).set_index('newid')

# Concatenate and drop any duplicate entries (keep the first occurrence)
news = pd.concat([train_news, dev_news])
news = news[~news.index.duplicated(keep='first')]

print(f"Combined news shape: {news.shape}")

Combined news shape: (65238, 7)


In [11]:
print(train_news['vertical'].nunique())

17


In [15]:
print(train_news['subvertical'].unique())

['lifestyleroyals' 'weightloss' 'newsworld' 'voices' 'medical'
 'football_nfl' 'weathertopstories' 'gaming' 'newsscienceandtechnology'
 'nutrition' 'autosenthusiasts' 'wellness' 'health-news' 'celebrity'
 'travelarticle' 'autossuvs' 'newspolitics' 'traveltripideas' 'autosnews'
 'newsbusiness' 'golf' 'lifestylepetsanimals' 'recipes' 'tv-gallery'
 'basketball_nba' 'lifestylebuzz' 'shop-all' 'newsphotos'
 'basketball_ncaa' 'finance-real-estate' 'quickandeasy' 'tv-celebrity'
 'travelnews' 'movies-gallery' 'tipsandtricks' 'autosbuying' 'more_sports'
 'shop-apparel' 'autostrucks' 'lifestyledidyouknow' 'racing' 'newstrends'
 'restaurantsandnews' 'lifestylemindandsoul' 'baseball_mlb'
 'finance-saving-investing' 'viral' 'finance-taxes' 'lifestylebeauty'
 'newsopinion' 'finance-companies' 'lifestyleshopping' 'finance-savemoney'
 'mentalhealth' 'newsus' 'lifestylesmartliving' 'fitness' 'autosclassics'
 'news' 'finance-career' 'lifestylehoroscope' 'newsgoodnews'
 'football_ncaa' 'retirement' 'life

In [26]:
# 2.2) Align news rows to LightFM item order
item_map = dataset.mapping()[2]  # article_id → row_idx
article_ids = [aid for aid,_ in sorted(item_map.items(), key=lambda x: x[1])]
news = news.reindex(article_ids).fillna('')


In [28]:
# 4.2) One-hot encode the 'vertical' column
vert_ohe = pd.get_dummies(news['vertical'], prefix='vert')
vert_matrix = csr_matrix(vert_ohe.values)


In [30]:
# 4.3) Sanity checks
assert vert_matrix.shape[0] == len(article_ids), "OHE row count mismatch"
row_sums = np.array(vert_matrix.sum(axis=1)).ravel()
assert (row_sums == 1).all(), f"{np.sum(row_sums!=1)} rows have invalid OHE"
print(f"Built vert_matrix: {vert_matrix.shape}, nnz={vert_matrix.nnz}")


Built vert_matrix: (9100, 16), nnz=9100


In [32]:
# ── A) Load your English BERT embeddings for both train and dev ──
emb_train = torch.load("english_article_embeddings.pt")
with open("english_news_ids.pkl","rb") as f: ids_train = pickle.load(f)


/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_46263/2502342605.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_train = torch.load("english_article_embed

In [34]:
# 5.2) Load dev embeddings and IDs
emb_dev = torch.load('english_article_embeddings_val.pt')
ids_dev = pickle.load(open('english_news_ids_val.pkl','rb'))


/var/folders/rw/b92f531j0kg9tv8t62n84v9c0000gn/T/ipykernel_46263/79931779.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb_dev = torch.load('english_article_embedding

In [36]:
# 5.3) Merge into full unique list
full_ids = []
full_vecs = []
for ids, embs in [(ids_train, emb_train), (ids_dev, emb_dev)]:
    for aid, vec in zip(ids, embs):
        if aid not in full_ids:
            full_ids.append(aid)
            full_vecs.append(vec.numpy())
full_vecs = np.vstack(full_vecs)

print(f"Full embeddings: {full_vecs.shape[0]} articles × {full_vecs.shape[1]} dims")


Full embeddings: 65238 articles × 384 dims


In [37]:
print(f"Full embeddings: {full_vecs.shape[0]} articles × {full_vecs.shape[1]} dims")


Full embeddings: 65238 articles × 384 dims


In [38]:
# 6.1) Allocate dense matrix aligned to item_map
n_items, bert_dim = len(item_map), full_vecs.shape[1]
dense_bert = np.zeros((n_items, bert_dim), dtype=np.float32)
for aid, vec in zip(full_ids, full_vecs):
    idx = item_map.get(aid)
    if idx is not None:
        dense_bert[idx] = vec


In [39]:
# 6.2) Dimensionality reduction via SVD
svd = TruncatedSVD(n_components=50, random_state=42)
bert_reduced = svd.fit_transform(dense_bert)
bert_features = csr_matrix(bert_reduced)
del dense_bert, bert_reduced

In [40]:
print(f"Compressed BERT features: {bert_features.shape}, nnz={bert_features.nnz}")


Compressed BERT features: (9100, 50), nnz=455000


In [41]:
# 7.1) Horizontal stack categorical + BERT features
item_features = hstack([vert_matrix, bert_features], format='csr')

# 7.2) Ensure no all-zero feature rows
zero_rows = np.where(item_features.getnnz(axis=1) == 0)[0]
assert len(zero_rows) == 0, f"{len(zero_rows)} items with no features"
print(f"Final item_features: {item_features.shape}, nnz={item_features.nnz}")


Final item_features: (9100, 66), nnz=464100


In [48]:
# A) item_features rows match number of items in the matrix
assert train_interactions.shape[1] == item_features.shape[0], (
    f"Items in train_interactions ({train_interactions.shape[1]}) != "
    f"feature rows ({item_features.shape[0]})"
)

# B) every item the model sees has at least one nonzero feature
zero_rows = np.where(item_features.getnnz(axis=1) == 0)[0]
assert len(zero_rows) == 0, f"{len(zero_rows)} items have all-zero features!"

In [50]:
import numpy as np
import random

# A) Every user in the interaction matrix is known to the dataset
n_users, n_items = train_interactions.shape
uid_map, iid_map = dataset.mapping()[0], dataset.mapping()[2]

# 1.1) Check that user indices span [0, n_users)
assert set(uid_map.values()) == set(range(n_users)), "Some user indices are missing or out of range"

# 1.2) Check that item indices span [0, n_items)
assert set(iid_map.values()) == set(range(n_items)), "Some item indices are missing or out of range"

print("✔ User/item index ranges are contiguous and complete")

# B) All (user,item) pairs in train/val interactions are valid IDs
#    We invert the maps to go back to IDs and sample a few
inv_uid = {v:k for k,v in uid_map.items()}
inv_iid = {v:k for k,v in iid_map.items()}

# 2.1) Sample some nonzero entries from train_interactions
coo = train_interactions.tocoo()
idxs = random.sample(range(coo.nnz), 5)
for k in idxs:
    uidx, iidx = coo.row[k], coo.col[k]
    uid, iid = inv_uid[uidx], inv_iid[iidx]
    # these should exist in your original DataFrames
    assert uid in all_users, f"Train user {uid} not in original users list"
    assert iid in all_items, f"Train item {iid} not in original items list"
print("✔ Sampled train interactions map back to valid user/article IDs")

# C) Feature‐interaction overlap
#    For each item in interactions, ensure at least one nonzero feature
item_nnz = np.array(item_features.getnnz(axis=1))
missing_feat_items = np.where(item_nnz == 0)[0]
assert len(missing_feat_items) == 0, f"{len(missing_feat_items)} interaction items have no features"

print("✔ Every item with interactions has at least one feature")

# D) Distributional spot‐check on feature norms
feat_norms = np.sqrt(item_features.multiply(item_features).sum(axis=1)).A1
print("Feature norms (min, mean, max):", np.min(feat_norms), np.mean(feat_norms), np.max(feat_norms))

# E) (Optional) A tiny “predict‐zero‐shot” test: recommend 1 item for a known user
#    and ensure you get back a valid article ID
from lightfm import LightFM
from lightfm.data import Dataset

# Re‐instantiate a minimal model for speed
model_test = LightFM(no_components=10, loss='warp')
model_test.fit_partial(train_interactions, item_features=item_features, epochs=1, num_threads=1)

# Pick a random user index
test_uidx = random.choice(list(uid_map.values()))
scores = model_test.predict(test_uidx, np.arange(n_items), item_features=item_features)
rec_iidx = np.argmax(scores)
print("Example rec for user", inv_uid[test_uidx], "→ article", inv_iid[rec_iidx])
assert inv_iid[rec_iidx] in all_items

print("✔ Quick one‐item recommendation test passed")


✔ User/item index ranges are contiguous and complete
✔ Sampled train interactions map back to valid user/article IDs
✔ Every item with interactions has at least one feature
Feature norms (min, mean, max): 2.028679 3.1985729 5.2614536
Example rec for user U61551 → article N49279
✔ Quick one‐item recommendation test passed


### Model Training

In [52]:
from lightfm import LightFM
from lightfm.evaluation import precision_at_k


In [54]:
model = LightFM(
    no_components=50,
    loss='warp'
)

In [56]:
model.fit(
    train_interactions,
    item_features=item_features,
    epochs=20,
    num_threads=4
)

In [57]:
# 3c) Evaluate Precision@10 on train and val
train_prec = precision_at_k(
    model,
    train_interactions,
    item_features=item_features,
    k=10
).mean()


In [58]:
val_prec = precision_at_k(
    model,
    val_interactions,
    train_interactions=train_interactions,  # mask out seen train items
    item_features=item_features,
    k=10
).mean()


In [59]:
print(f"Precision@10 → train: {train_prec:.4f}, validation: {val_prec:.4f}")

Precision@10 → train: 0.1091, validation: 0.0002


### Saving learned latent representations

In [61]:
user_latents, _ = model.get_user_representations()
item_latents, _ = model.get_item_representations()

with open('user_latents.pkl', 'wb') as f:
    pickle.dump(user_latents, f)
with open('item_latents.pkl', 'wb') as f:
    pickle.dump((item_latents, dataset.mapping()[2]), f)

print("Saved user and item latents.")

Saved user and item latents.
